# Objective <br>
Determine whether the profitability problem in Tables and Bookcases is concentrated in particular regions or widespread across all four regions.

In [1]:
import os
import pandas as pd

# List files in the current working directory
print("Current directory:", os.getcwd())
print("\nAvailable files:")

for file in os.listdir():
    print(file)



df = pd.read_csv("../data/Sample - Superstore.csv", encoding="cp1252")

print("Dataset loaded successfully")
print("Shape:", df.shape)

display(df.head())

Current directory: d:\consulting\Retail Profitability Turnaround\notebook

Available files:
01_load_data.ipynb
02_analyis.ipynb
03_Root-Cause Analysis of Profitability Gaps.ipynb
04_Product_Region_Profitability.ipynb
Dataset loaded successfully
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [2]:
target_products = df[
    df["Sub-Category"].isin(["Tables", "Bookcases"])
].copy()

product_region_analysis = (
    target_products
    .groupby(["Sub-Category", "Region"])
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Records=("Profit", "size"),
        Loss_Making_Records=("Profit", lambda x: (x < 0).sum()),
        Avg_Discount=("Discount", "mean")
    )
    .reset_index()
)

product_region_analysis["Profit_Margin_%"] = (
    product_region_analysis["Profit"]
    / product_region_analysis["Sales"] * 100
)

product_region_analysis["Loss_Record_Share_%"] = (
    product_region_analysis["Loss_Making_Records"]
    / product_region_analysis["Records"] * 100
)

display(
    product_region_analysis
    .sort_values(["Sub-Category", "Profit"])
    .round(2)
)

,Sub-Category,Region,Sales,Profit,Records,Loss_Making_Records,Avg_Discount,Profit_Margin_%,Loss_Record_Share_%
0,Bookcases,Central,24157.18,-1997.90,50,36,0.23,-8.27,72.00
3,Bookcases,West,36004.12,-1646.51,80,32,0.23,-4.57,40.00
1,Bookcases,East,43819.33,-1167.63,70,33,0.22,-2.66,47.14
2,Bookcases,South,10899.36,1339.49,28,8,0.10,12.29,28.57
5,Tables,East,39139.81,-11025.38,80,78,0.37,-28.17,97.50
6,Tables,South,43916.19,-4623.06,51,27,0.22,-10.53,52.94
4,Tables,Central,39154.97,-3559.65,72,49,0.26,-9.09,68.06
7,Tables,West,84754.56,1482.61,116,49,0.20,1.75,42.24


Interpretation:
- Central has the lowest margin and the highest share of loss-making Bookcases records.
- West and East also lose money overall.
- South is profitable, with a 12.29% margin and a lower loss-making record share.

Interpretation:
- East accounts for a loss of $11,025.38 on Tables, with almost every record loss-making.
- Its average discount is 37%, substantially higher than the other regions.
- West is the only region with positive aggregate profit on Tables, despite 42.24% of its records being loss-making.
- South and Central are also loss-making, but their margins are less negative than East's.
This is a strong descriptive pattern: East combines the highest average discount with the worst Table profitability. It supports investigating discount practices in East first, but it does not prove discounts alone explain the regional difference.

Regional differences suggest different intervention priorities. Tables in the East and Bookcases in Central should receive priority for further investigation. The West for Tables and the South for Bookcases provide useful comparison cases.

Business implication: Investigate regional discounting and product mix before recommending pricing changes. The observed association between discounts and losses does not establish causation, and regional differences may also reflect product selection, costs, or order characteristics.